In [ ]:
import geopandas as gpd
import pandas as pd
from project_paths import paths

In [ ]:
aims = gpd.read_file(paths.aims_data / "aims.gpkg")
aims

In [ ]:
aims.crs

In [ ]:
centroids_df = gpd.GeoDataFrame(aims["asset_id"], geometry=aims.geometry.centroid, crs=aims.crs)
centroids_df

In [ ]:
eir_df = pd.read_csv(paths.ea_data / "ea_eir_data.csv")
eir_df

In [ ]:
print(len(eir_df["Asset AIMS ID"].unique()))
print(len(eir_df["Asset AIMS ID"]))

In [ ]:
import numpy as np

def convert(series: pd.Series) -> pd.Series:
    num = pd.to_numeric(series, errors="coerce")
    num_cleaned = num.where(~num.isin([888, 999]), other=np.nan)
    return num_cleaned

In [ ]:
eir_df["Year Last Refurbished"] = convert(eir_df["Year Last Refurbished"])
eir_df["Asset Year Last Refurbished"] = convert(eir_df["Asset Year Last Refurbished"])

In [ ]:
print(eir_df[["Asset AIMS ID", "Year Last Refurbished", "Asset Year Last Refurbished"]].describe())
print()
print(eir_df[["Asset AIMS ID", "Year Last Refurbished", "Asset Year Last Refurbished"]].info())

In [ ]:
both_but_different = eir_df[eir_df["Year Last Refurbished"].notna() & eir_df["Asset Year Last Refurbished"] & (eir_df["Year Last Refurbished"] != eir_df["Asset Year Last Refurbished"])]
both_but_different

good


In [ ]:
only_year = eir_df[eir_df["Year Last Refurbished"].notna() & eir_df["Asset Year Last Refurbished"].isna()]

only_asset_year = eir_df[eir_df["Year Last Refurbished"].isna() & eir_df["Asset Year Last Refurbished"].notna()]

print(len(only_year))
print(len(only_asset_year))

In [ ]:
eir_df["raw Calculated Condition"] = eir_df["Calculated Condition"]
eir_df["Calculated Condition"] = convert(eir_df["Calculated Condition"])

print(len(eir_df[eir_df["raw Calculated Condition"].eq(888)]))
print(len(eir_df[eir_df["raw Calculated Condition"].eq(999)]))
print(len(eir_df[eir_df["raw Calculated Condition"].isin([0,1,2,3,4,5])]))
print(len(eir_df[eir_df["raw Calculated Condition"].isna()]))

print(len(eir_df[eir_df["Calculated Condition"].isna()]))      
print(len(eir_df[eir_df["Calculated Condition"].notna()]))      


todo - EA say in response that the redaction is by asset, so there should be no asset ids where there is both an na and notna condition grade. 

In [ ]:
eir_clean_df = eir_df[eir_df["Calculated Condition"].notna()].copy()
eir_clean_df.shape

In [ ]:
# currently the eir data has multiple entries per asset for multiple inspections
# so currently longitudinal.
# this will be useful for later, but for now i want a cross sectional dataset for the first steps

eir_clean_df["inspection_date"] = pd.to_datetime(eir_clean_df["Completion Date"], errors="coerce", format="%d/%m/%Y")
print(eir_clean_df.head())
print()
print(eir_clean_df["inspection_date"].isna().sum())

In [ ]:
print(eir_clean_df[eir_clean_df["inspection_date"].isna()])

it ought to be impossible to have a grade without an inspection date?

In [ ]:
print(eir_clean_df[eir_clean_df["inspection_date"].isna()][["Asset AIMS ID", "Calculated Condition"]].groupby("Calculated Condition").count())

decision point - should drop these rows? 



In [ ]:
eir_cs = (
    eir_clean_df
    .sort_values(by = "inspection_date", ascending=False, na_position="last")
    .drop_duplicates(subset="Asset AIMS ID", keep="first")
)

print(eir_cs.info())
print()
print(eir_cs.describe())
print()
print(eir_cs.head())

distribution of grades looks extremely unbalanced

In [ ]:
eir_cs = eir_cs.rename({
    "Asset AIMS ID": "asset_id",
    "Completion Date": "eir_completion_date",
    "Calculated Condition": "eir_condition_grade",
    "Asset FCRM Target Condition": "eir_asset_target_condition",
    "BRC": "eir_below_required_condition",
    "Year Last Refurbished": "eir_year_last_refurbished",
    "Repair Deadline Date": "eir_repair_deadline",
    "Agreed by Designated Engineer": "eir_agreed_by_engineer",
    "Asset Year Last Refurbished": "eir_asset_year_last_refurbished",
    "raw Calculated Condition": "eir_raw_calculated_condition",
    "inspection_date": "eir_inspection_date",
}, axis="columns")

print(eir_cs.columns)

eir_cs = eir_cs[
    [
        "asset_id",
        "eir_inspection_date",
        "eir_condition_grade",
        "eir_asset_target_condition",
        "eir_below_required_condition",
        "eir_year_last_refurbished",
        "eir_repair_deadline",
        "eir_agreed_by_engineer",
    ]
]


eir_cs

In [ ]:
eir_cs_clean = eir_cs.copy()

target_condition_map = {
    str(grade).split(" - ")[0]: str(grade).split(" - ")[1]
    for grade in eir_cs_clean["eir_asset_target_condition"].unique().tolist()
    if grade is not np.nan and len(str(grade).split(" - ")) == 2
}

print(target_condition_map)


In [ ]:

eir_cs_clean["eir_asset_target_condition"] =  list(map(lambda x: str(x).split(" - ")[0], eir_cs_clean["eir_asset_target_condition"]))

eir_cs_clean

In [ ]:
# joined_aims_eir = pd.DataFrame.join(aims, eir_cs_clean, on="asset_id", how="inner", validate="1:1")
# above - mismatched types - asset id is numeric in one df and string in another.
# should have a cell that specifies the schema of each df explicitly, like the naming one for eir

joined_aims_eir = pd.concat([aims.set_index("asset_id"), eir_cs_clean.set_index("asset_id")], axis=1, join="outer", verify_integrity=True)

print(joined_aims_eir.head())
print()
print(joined_aims_eir.info())
print()
print(joined_aims_eir.describe())

In [ ]:
condition_coverage_check = joined_aims_eir[
    (joined_aims_eir["current_condition"].isin([" ", "", "\t"]) | joined_aims_eir["current_condition"].isna()) & joined_aims_eir["eir_condition_grade"].notna()
]

condition_coverage_check

looks like join failed completely

probably the type mismatch on asset id

In [ ]:
aims.dtypes

In [ ]:
eir_cs_clean.dtypes

In [ ]:
aims["asset_id"] = aims["asset_id"].astype("int64")

aims.dtypes

In [ ]:
aims.reset_index(inplace=True)
eir_clean_df.reset_index(inplace=True)

joined_aims_eir = aims.merge(eir_cs_clean, on="asset_id", how="inner", validate="1:1")

In [ ]:
joined_aims_eir

In [ ]:
print(joined_aims_eir.head())
print()
print(joined_aims_eir.info())
print()
print(joined_aims_eir.describe())

In [ ]:
condition_coverage_check = joined_aims_eir[
    (joined_aims_eir["current_condition"].isin([" ", "", "\t"]) | joined_aims_eir["current_condition"].isna()) & joined_aims_eir["eir_condition_grade"].notna()
]

condition_coverage_check

In [ ]:
condition_coverage_check[["asset_id", "asset_name", "asset_sub_type", "current_condition", "eir_condition_grade"]]

In [ ]:
condition_agreement_check = joined_aims_eir[
    (joined_aims_eir["current_condition"].replace(r"^\s*$", np.nan, regex=True).astype("float32") != joined_aims_eir["eir_condition_grade"].astype("float32"))
]

condition_agreement_check[["asset_id", "asset_name", "asset_sub_type", "current_condition", "eir_condition_grade"]]

I can immediately see that there are discrepencies in both directions - 4 (higher) in the aims download vs 3 (lower) in the EIR response, and 3 (lower) in the aims download vs 4 (higher) in the EIR response. 

My instinct at the moment is that the EIR grade should be more trustworthy because i also have the exact inspection date. I expect the discrepencies are due to recent inspections. It may be worth redownloading aims and seeing if it agrees more closely with the eir response.

---

bgs

---


In [ ]:



geology_path = paths.geology_data / "625k_V5_Geology_UK_EPSG27700.gpkg"

In [ ]:
import fiona 

layers = fiona.listlayers(geology_path)

layers

In [ ]:
bedrock_layer = layers[0]

superficial_geo_layer = layers[3]



In [ ]:
bedrock_gdf = gpd.read_file(geology_path, layer=bedrock_layer)

superficial_geo_gdf = gpd.read_file(geology_path, layer=superficial_geo_layer)

In [ ]:
bedrock_gdf.describe()

In [ ]:
superficial_geo_gdf.describe()

In [ ]:
bedrock_gdf.head()

In [ ]:
superficial_geo_gdf.head()

i need to find a data description/dictionary/info doc to interpret these fields. There was one in the download file for the geosure dataset, but not for this one it seems. 

In [ ]:
bedrock_gdf.columns

https://www.bgs.ac.uk/technologies/the-bgs-lexicon-of-named-rock-units/

In [ ]:
geology_columns = [
    "LEX",
    "LEX_D",
    "LEX_RCS",
    "RCS",
    "RCS_X",
    "RCS_D",
    "MAX_PERIOD", 
    "MAX_ERA",
    "BED_EQ",
    "BED_EQ_D",
]


bedrock_gdf[geology_columns]

anything with a "_D" suffix is a human readable description.



ive found data descriptions for the 50k dataset, but not yet for the 625k dataset (the one im actually using). I think its safe to assume these column descriptions are transferrable for now. 

https://earthwise.bgs.ac.uk/index.php/OR/16/046_Technical_information


lex = lexicon
rcs = rock classification code
rank = "Rank of the unit in the lithostratigraphical or lithodemic hierarchy: e.g. BED or SUITE"
...


looks like theres a few different types of information, but several columns are going to contain overlapping signal. most obvious example is the _D descriptions - exactly the same info as without the suffix but displayed differently. Also, _X suffixes appear to stand for extended or expanded - all the information as the non suffixed column plus something extra, e.g. with the rcs rock code, its the same rock code plus some other codes.

So obviously this will be important for feature selection, either only selecting the column with the most signal or combining some of them into single features. But I don't think i have enough context to decide which columns this will be at the moment, so i will keep them all in the join and deal with them in another notebook

In [ ]:
bedrock_gdf[["RCS", "RCS_X"]]

In [ ]:
bedrock_gdf["RCS"].unique()

In [ ]:
bedrock_gdf["RCS_X"].unique()

In [ ]:
joined_aims_eir.head()

In [ ]:
print(centroids_df.crs)

centroids_df["asset_id"] = centroids_df["asset_id"].astype("int64")

joined_geology_centroids = gpd.sjoin(
    centroids_df,
    bedrock_gdf.to_crs("EPSG:27700"),
    how="left",
    predicate="within"
)

# joined_geology_centroids

joined_aims_eir_geology = joined_aims_eir.merge(
    right=joined_geology_centroids,
    how="left",
    on="asset_id",
    suffixes=("", "_bedrock")
)

joined_aims_eir_geology

In [ ]:
# superficial_geo_gdf

joined_super_centroids = gpd.sjoin(
    centroids_df,
    superficial_geo_gdf.to_crs("EPSG:27700"),
    how="left",
    predicate="within"
)

# joined_geology_centroids

joined_aims_eir_geology = joined_aims_eir_geology.merge(
    right=joined_super_centroids,
    how="inner",
    on="asset_id", 
    suffixes=("", "_superficial")
)

joined_aims_eir_geology

In [ ]:
n_dupes = joined_aims_eir_geology["asset_id"].duplicated().sum()
n_dupes

nice

In [ ]:

print(joined_aims_eir_geology["index"].isna().sum())

nice

In [ ]:
geosure_files = [file for file in paths.geosure_data.glob("GB_Hex_5km_GS_*_v8.shp")]

geosure_files

In [ ]:
test = gpd.read_file(geosure_files[3])
test

confirmed all files have the same columns

only the CLASS column is going to be significantly useful, possible that the advisory and notice columns can also be used.
But the legend is fully functionally dependednt on the class, and the version will always be v8 and has no signal.

In [ ]:
named_files = {
    filepath.stem.replace("GB_Hex_5km_GS_", "").replace("_v8", ""): filepath
    for filepath in geosure_files
}



In [ ]:
import re


spine = centroids_df.copy()


def camel_to_snake(name):
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', name).lower()


for file in geosure_files:

    column_suffix = f"_geosure_{camel_to_snake(file.stem.replace("GB_Hex_5km_GS_", "").replace("_v8", ""))}"
    print(column_suffix)

    gdf = gpd.read_file(file)
    gdf.set_crs(epsg=27700)

    spine = spine.sjoin(
        gdf[[
            "CLASS", "Advisory", "Notice", "geometry"
        ]],
        how="left",
        predicate="within",
        rsuffix=column_suffix
    )

    spine = spine.drop(columns=["index_right"], errors="ignore")

spine.columns

In [ ]:
import re


spine = centroids_df.copy()


def camel_to_snake(name):
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', name).lower()


for file in geosure_files:
    
    column_suffix = f"geosure_{camel_to_snake(file.stem.replace("GB_Hex_5km_GS_", "").replace("_v8", ""))}"
    print(column_suffix)
    
    gdf = gpd.read_file(file).set_crs(epsg=27700, allow_override=True)
    
    joined = gpd.sjoin(
        centroids_df[["asset_id", "geometry"]],
        gdf[["CLASS", "Advisory", "Notice", "geometry"]],
        how="left",
        predicate="within",
    ).drop(columns=["index_right"], errors="ignore")
    
    joined = joined.rename(columns={
        "CLASS": f"{column_suffix}__class",
        "Advisory": f"{column_suffix}__advisory",
    })
    
    spine = spine.merge(
        joined[["asset_id", f"{column_suffix}__class", f"{column_suffix}__advisory"]],
        on="asset_id",
        how="left",
    )

spine

In [ ]:
gdf_geosure = spine.copy()

In [ ]:
cols = gdf_geosure.columns.to_list()
cols.remove("geometry")

joined_aims_eir_geology_gs = joined_aims_eir_geology.merge(
    right=gdf_geosure[cols],
    on="asset_id",
    how="left"
)

joined_aims_eir_geology_gs

In [ ]:
print(joined_aims_eir_geology_gs["asset_id"].duplicated().sum())
print(joined_aims_eir_geology_gs["index"].isna().sum())

---

bgs hydrology

---

In [ ]:
hydrogeo_gdf = gpd.read_file(paths.hydrogeology_data)

hydrogeo_gdf

In [ ]:
joined_hydrog_centroids = gpd.sjoin(
    centroids_df,
    hydrogeo_gdf.to_crs("EPSG:27700"),
    how="left",
    predicate="within"
)

joined_aims_eir_geology_gs_hydrogeo = joined_aims_eir_geology_gs.merge(
    joined_hydrog_centroids[["asset_id", "ROCK_UNIT", "CLASS", "CHARACTER", "FLOW_MECHA", "SUMMARY"]],
    how="left",
    on="asset_id",
    suffixes=("", "_hydrog")
)

joined_aims_eir_geology_gs_hydrogeo.head()

In [ ]:
print(joined_aims_eir_geology_gs_hydrogeo["asset_id"].duplicated().sum())
print(joined_aims_eir_geology_gs_hydrogeo["index"].isna().sum())

thus all data is joined.

next, inspect dataframe, drop any columns that are clearly duplicate info or purely admin/lookup, then rename everything to follow a sensible naming convention including the source, and maybe coherse some datatypes. 

then save it

In [ ]:
joined_aims_eir_geology_gs_hydrogeo.columns.to_list()

In [ ]:
import polars as pl
import polars.selectors as cs

unified_aims_eir_bgs = pl.from_pandas(joined_aims_eir_geology_gs_hydrogeo.drop(columns=["geometry", "geometry_bedrock", "geometry_superficial"])).select(
        
        cs.by_name(
            'asset_id',
            'asset_name',
            'asset_sub_type',
            'primary_purpose',
            'alternative_purpose_1',
            'alternative_purpose_2',
            'asset_length',
            'toe_level',
            'protection_type',
            'asset_maintainer',
            'asset_operator',
            'asset_owner',
            'current_condition',
            'target_condition',
            'current_sop',
            'current_sop_date',
            'design_sop',
            'design_sop_dqf',
            'design_ucl',
            'year_last_refurbished',
            'last_inspection_date',
            'next_inspection_date',
            'asset_start_date',
            'actual_dcl',
            'actual_dcl_dqf',
            'actual_ucl',
            'actual_ucl_dqf',
            'design_dcl',
            'bank',
            'effective_cl',
            'effective_cl_dqf',
            'include_in_flood_map',
            'water_management_area',
            'local_authority',
            'frms_code',
            'water_course_name',
        ).name
        .prefix("aims__"),

        cs.starts_with("eir_")
        .name
        .map(lambda name: "eir__" + name.removeprefix("eir_")),

        cs.starts_with("geosure_"),
        
        cs.by_name("ROCK_UNIT", "CLASS", "CHARACTER", "FLOW_MECHA", "SUMMARY")
        .name
        .map(lambda n: "hydrogeo__" + n.lower()),
        
        cs.by_name(
            "LEX",
            "LEX_D",
            "LEX_RCS",
            "RCS",
            "RCS_X",
            "RCS_D",
            "RANK",
            "BED_EQ",
            "BED_EQ_D",
            "MB_EQ",
            "MB_EQ_D",
            "FM_EQ",
            "FM_EQ_D",
            "SUBGP_EQ",
            "SUBGP_EQ_D",
            "GP_EQ",
            "GP_EQ_D",
            "SUPGP_EQ",
            "SUPGP_EQ_D",
            "MAX_TIME_D",
            "MIN_TIME_D",
            "MAX_TIME_Y",
            "MIN_TIME_Y",
            "MAX_INDEX",
            "MIN_INDEX",
            "MAX_AGE",
            "MIN_AGE",
            "MAX_EPOCH",
            "MIN_EPOCH",
            "MAX_SUBPER",
            "MIN_SUBPER",
            "MAX_PERIOD",
            "MIN_PERIOD",
            "MAX_ERA",
            "MIN_ERA",
            "MAX_EON",
            "MIN_EON",
            "PREV_NAME",
            "LEX_RCS_D",
        ).name
        .map(lambda name: "bedrock_geo__" + name.removesuffix("_bedrock").lower()),

        (
            (cs.ends_with("_superficial") | cs.by_name(
                    "LEX_ROCK",
                    "ROCK",
                    "ROCK_D",
                    "MAX_AGE_NO",
                    "MIN_AGE_NO",
                    "MAX_STAGE",
                    "MIN_STAGE",
                    "MAX_SERIES",
                    "MIN_SERIES",
                    "MAX_SUBSYS",
                    "MIN_SUBSYS",
                    "MAX_SYSTEM",
                    "MIN_SYSTEM",
                    "MAX_ERATH",
                    "MIN_ERATH",
                    "MAX_EONOTH",
                    "MIN_EONOTH",
                ) 
            ) - cs.matches(
                    r"(?i)bgsref|mslink|nom_scale|nom_os_yr|nom_bgs_yr|sheet|version|released|lex_rcs_i|map_code|age_onegl|bgstype|index_right|geometry"
                )
        )
        .name
        .map(lambda name: "superficial_geo__" + name.removesuffix("_superficial").lower()),

)



unified_aims_eir_bgs

In [ ]:
unified_aims_eir_bgs.columns

so far, columns have been removed if:

- they contain geometry. if this is needed later for further joins, the pandas dataframe can be recovered. I doon't think i want this as a feature as i don't want a model to simply learn a geographic pattern i.e. all flood defenses in the south east are better maintained. (spec)

- they definitly contain no signal (for predictive purposes). This includes the index, the bgs refs, versions, and release dates, as well as some of the markers like the "BEDROCK"/"SUPERFICIAL" flag which has moved to the column name. 

At this stage, there are still lots of columns that I wouldn't expect to survive eda or will get removed in feature selection because, either:

1. they are too sparse - for example a lot of the columns coming from the superficial geology appear to be entirely null
2. the signal overlaps with another column
3. they are deemed to contain no valuable predictive signal - this is the same as the second point above, but i would consider the removal i have done so far to be cautious, and there are other columns such as water course name that i haven't removed yet while it might still be interesting as a descriptor, and others like "eq" where i haven't yet decided how valuable it is.

In [ ]:
unified_aims_eir_bgs.schema

In [ ]:
unified_aims_eir_bgs.write_parquet(paths.processed_data / "unified_aims_eir_bgs.parquet")